# scRareBench | High-Level MrVI | Dataset 2 — mBDRC renal cortex

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook runs **MrVI** on scRareBench Dataset 2 (`mbdrc_renal_cortex`) through the generic **MethodSpec → benchmark_method** API. MrVI remains user-owned method code; scRareBench owns dataset loading, seed orchestration, latent alignment, evaluation, and reporting.

### Fixed Dataset 2 / MrVI contract
- Dataset selector: `2` / `mbdrc_renal_cortex`
- Benchmark label: registered Dataset 2 label (`cell_type`), **never supplied to MrVI**
- Evaluation batch: registered `donor_id × assay` combination
- MrVI `sample_key`: `donor_id`
- MrVI nuisance `batch_key`: `assay`
- Method preprocessing: **global Seurat-v3 4000 HVGs from raw counts**, with no batch key in HVG selection
- `n_latent_u=30`, `n_latent_z=30`, encoder hidden `128`, encoder layers `2`
- Training: max `300` epochs, train/validation `0.90/0.10`, batch size `128`, early-stopping patience `25`
- Benchmark representation: posterior-mean **u latent**, `give_z=False`
- Method seeds: `42, 123, 2026`; benchmark seed: `42`

> Use a **GPU + High-RAM Colab runtime**. For a smoke test, temporarily set `METHOD_SEEDS = [42]`.

## 1. Install scRareBench and scvi-tools / MrVI

In [ ]:
import subprocess, sys

SCRAREBENCH_GITHUB = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB])

from scrarebench.runtime import print_install_report, setup_runtime
install_report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("scvi", "scvi.external"),
    quiet=False,
)
print_install_report(install_report)

import scvi
from scvi.external import MRVI
print("scvi-tools:", scvi.__version__)
print("MRVI import: OK")

In [ ]:
# OPTIONAL runtime diagnostics only.
# import sys, torch
# from importlib import metadata
# print("Python:", sys.version)
# print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
# for pkg in ("numpy", "scanpy", "anndata", "scvi-tools"):
#     try: print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError: print(pkg, "not installed")

## 2. Load Dataset 2 / mBDRC

In [ ]:
from scrarebench import load_dataset, dataset_info

adata = load_dataset(2)
info = dataset_info(adata)
print(adata)
print(info)

if info.get("label_key") is None or info.get("batch_key") is None:
    raise RuntimeError(f"Dataset 2 benchmark metadata is incomplete: {info}")
if info.get("scib_hvg_batch_mode") != "global":
    raise RuntimeError(f"Dataset 2 must use global HVG selection: {info}")
for key in ("donor_id", "assay"):
    if key not in adata.obs.columns:
        raise KeyError(f"Dataset 2 is missing required MrVI metadata {key!r}")
if not adata.obs_names.is_unique:
    raise ValueError("Dataset 2 observation names must be unique")

print("Benchmark label:", info["label_key"])
print("Evaluation batch:", info["batch_key"])
print("MrVI sample key: donor_id")
print("MrVI nuisance batch key: assay")

## 3. Recover and validate raw counts

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp

def _diag(matrix, max_values=100_000, seed=42):
    v = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    if len(v) > max_values:
        rng=np.random.default_rng(seed); v=v[rng.choice(len(v), max_values, replace=False)]
    v=np.asarray(v,dtype=float); finite=v[np.isfinite(v)]
    return {
        "finite_fraction": float(np.mean(np.isfinite(v))) if len(v) else 1.0,
        "nonnegative": bool(len(finite)==0 or finite.min()>=0),
        "integer_like": bool(len(finite)==0 or np.allclose(finite,np.rint(finite),atol=1e-6)),
    }

def _copy(matrix):
    return matrix.copy().tocsr() if sp.issparse(matrix) else np.asarray(matrix).copy()

def ensure_counts_layer(adata, key="counts"):
    if key in adata.layers:
        d=_diag(adata.layers[key])
        if d["nonnegative"] and d["integer_like"]: return f"layers/{key}", d
        raise ValueError(f"Existing {key!r} layer is not raw counts: {d}")
    if adata.raw is not None:
        raw_names=pd.Index(adata.raw.var_names.astype(str)); cur=pd.Index(adata.var_names.astype(str))
        pos=raw_names.get_indexer(cur)
        if np.all(pos>=0):
            matrix=adata.raw.X[:,pos]; d=_diag(matrix)
            if d["nonnegative"] and d["integer_like"]:
                adata.layers[key]=_copy(matrix); return "raw.X aligned to adata.var_names", d
    d=_diag(adata.X)
    if d["nonnegative"] and d["integer_like"]:
        adata.layers[key]=_copy(adata.X); return "X", d
    raise ValueError(f"Could not locate raw counts. layers={list(adata.layers)}, raw={adata.raw is not None}, X={d}")

COUNT_LAYER = info.get("count_layer") or "counts"
count_source, count_diagnostics = ensure_counts_layer(adata, COUNT_LAYER)
print("Raw-count source:", count_source)
print("Count diagnostics:", count_diagnostics)
if count_diagnostics["finite_fraction"] != 1.0: raise ValueError("Raw counts contain NaN/Inf")

## 4. Reproducibility and MrVI configuration

In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_policy": "global_seurat_v3_raw_counts",
    "hvg_batch_key": None,
    "hvg_span": 0.3,
    "sample_key": "donor_id",
    "nuisance_batch_key": "assay",
    "n_latent_u": 30,
    "n_latent_z": 30,
    "encoder_n_hidden": 128,
    "encoder_n_layers": 2,
    "max_epochs": 300,
    "train_size": 0.90,
    "batch_size": 128,
    "early_stopping": True,
    "early_stopping_patience": 25,
}

OUTPUT_DIR = "/content/scrarebench_mrvi_dataset2_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_mrvi_dataset2_method_runs"

## 5. User-owned MrVI runner

The runner uses raw counts, `donor_id`, and `assay` only. It never reads the benchmark label. The returned representation is the posterior-mean **u** latent (`give_z=False`).

In [ ]:
from pathlib import Path
import gc, random
import numpy as np, pandas as pd, scanpy as sc, torch, scvi
from scvi.external import MRVI
from scrarebench import MethodOutput

def run_mrvi(method_adata, seed, config):
    count_layer = info.get("count_layer") or "counts"
    sample_key = config["sample_key"]
    nuisance_key = config["nuisance_batch_key"]
    barcodes = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(barcodes)) != len(barcodes): raise ValueError("Duplicate Dataset 2 barcodes")
    for key in (sample_key, nuisance_key):
        if key is not None and key not in method_adata.obs.columns: raise KeyError(key)
    if count_layer not in method_adata.layers: raise KeyError(f"Missing raw-count layer {count_layer!r}")

    # Fixed Dataset 2 policy: GLOBAL Seurat-v3 HVGs from raw counts.
    hvg_kwargs = dict(
        layer=count_layer, flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        span=float(config["hvg_span"]), subset=False, check_values=True,
    )
    if config.get("hvg_batch_key") is not None:
        hvg_kwargs["batch_key"] = config["hvg_batch_key"]
    sc.pp.highly_variable_genes(method_adata, **hvg_kwargs)
    mask=method_adata.var["highly_variable"].fillna(False).to_numpy(dtype=bool)
    expected=min(int(config["n_hvg"]), method_adata.n_vars)
    if int(mask.sum()) != expected: raise RuntimeError(f"Expected {expected} HVGs, got {int(mask.sum())}")
    method_adata=method_adata[:,mask].copy()
    current=method_adata.obs_names.astype(str).to_numpy()
    if not np.array_equal(current,barcodes): raise RuntimeError("Cell order changed during HVG subsetting")

    seed=int(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    scvi.settings.seed=seed

    setup_kwargs={"sample_key": sample_key, "backend": "torch", "layer": count_layer}
    if nuisance_key is not None: setup_kwargs["batch_key"] = nuisance_key
    MRVI.setup_anndata(method_adata, **setup_kwargs)

    model=MRVI(
        method_adata,
        n_latent=int(config["n_latent_z"]),
        n_latent_u=int(config["n_latent_u"]),
        encoder_n_hidden=int(config["encoder_n_hidden"]),
        encoder_n_layers=int(config["encoder_n_layers"]),
    )
    train_size=float(config["train_size"])
    model.train(
        max_epochs=int(config["max_epochs"]), accelerator="auto", devices="auto",
        train_size=train_size, validation_size=1.0-train_size,
        batch_size=int(config["batch_size"]),
        early_stopping=bool(config["early_stopping"]),
        early_stopping_patience=int(config["early_stopping_patience"]),
        check_val_every_n_epoch=1, enable_progress_bar=True,
    )

    latent=np.asarray(model.get_latent_representation(
        adata=method_adata, batch_size=int(config["batch_size"]),
        use_mean=True, give_z=False,
    ),dtype=np.float32)
    expected_shape=(method_adata.n_obs,int(config["n_latent_u"]))
    if latent.shape != expected_shape: raise RuntimeError(f"Unexpected MrVI-u shape {latent.shape}; expected {expected_shape}")
    if not np.isfinite(latent).all(): raise ValueError("MrVI-u latent contains NaN/Inf")

    seed_dir=Path(METHOD_WORK_DIR)/f"seed_{seed}"; seed_dir.mkdir(parents=True,exist_ok=True)
    provenance={}
    frames=[]
    for metric, values in getattr(model,"history",{}).items():
        frame=pd.DataFrame(values).copy(); frame["metric"]=metric; frame["epoch"]=frame.index; frames.append(frame.reset_index(drop=True))
    if frames:
        history_path=seed_dir/"mrvi_training_history.csv"
        pd.concat(frames,ignore_index=True).to_csv(history_path,index=False)
        provenance["mrvi_training_history"] = history_path

    print(f"Seed {seed}: global HVGs={int(mask.sum())}; MrVI-u latent={latent.shape}")
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return MethodOutput(latent=latent, barcodes=current, representation_key="X_mrvi_u", provenance_files=provenance)

## 6. Run the High-Level multi-seed benchmark

For an initial smoke test, temporarily use `METHOD_SEEDS = [42]`, then restore all three seeds for the final benchmark.

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="MrVI-u",
    runner=run_mrvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata, method, seeds=METHOD_SEEDS, output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "include_latent_in_bundle": False,
    },
    install_dependencies=False,
    finalize=True,
)

## 7. Inspect the finalized result

In [ ]:
from IPython.display import display
display(result.summary().round(5))
print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)
for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  output:", run.output_dir)
    print("  interactive report:", run.interactive_report_path)
    print("  PDF:", run.pdf_path)
    print("  bundle:", run.bundle_path)

## 8. Download the final multi-seed archive

In [ ]:
from pathlib import Path
from google.colab import files
archive_path=Path(result.archive_path)
if not archive_path.exists():
    raise FileNotFoundError(f"Output archive was not found: {archive_path}")
print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size/(1024**2):.2f} MB")
files.download(str(archive_path))